# Загрузка данных и библиотек


## Импорт библиотек


In [1]:
import pandas as pd
import numpy as np
from collections import Counter

## Загрузка датасетов


In [2]:
# Загружаем основной датасет заявок после EDA
app = pd.read_csv('../data/app_train_eda.csv')

# Загружаем историю предыдущих заявок клиентов
prev = pd.read_csv('../data/previous_application.csv')

# Feature Engineering из previous_application

Самая важная информация о клиенте содержится не в датасете application.csv, а в прочих таблицах, определяющих поведенческие паттерны клиента до подачи заявки. Необходимо их правильно агрегировать, получить большое количество новых признаков и после отобрать из них самые полезные.


## Исследование предыдущих заявок

Рассмотрим, сколько заявок у клиентов бывало до подачи последней заявки, риск которой мы должны оценить


In [4]:
# Строим гистограмму распределения количества предыдущих заявок на клиента
prev.groupby('SK_ID_CURR')['SK_ID_PREV'].count().hist(bins=100)

Видно, что как минимум 60К человек ранее подавали заявку на кредит, из этого мы можем извлечь много нового


## Базовые признаки на основе предыдущих заявок


In [ ]:
# Если нет сведений о первоначальном взносе, то считаем, что он равен 0
prev['AMT_DOWN_PAYMENT'] = prev['AMT_DOWN_PAYMENT'].fillna(0)

# Определяем наиболее частый день недели подачи заявки для каждого клиента
app['WEEKDAY_APPR_PROCESS_START_most_freq'] = (
    prev
    .groupby('SK_ID_CURR')['WEEKDAY_APPR_PROCESS_START']
    .agg(lambda x: x.value_counts().idxmax())
)

### Количество предыдущих заявок


In [ ]:
# Считаем общее количество предыдущих заявок для каждого клиента
count_loan = Counter(prev['SK_ID_CURR'])
app['prev_loan_count'] = app['SK_ID_CURR'].map(count_loan).fillna(0).astype(int)

### Соотношения признаков внутри заявки

Добавляем относительные признаки на основе полей заявки


In [ ]:
# Отношение суммы кредита к запрашиваемой сумме
prev['credit_to_app_ratio'] = prev['AMT_CREDIT'] / prev['AMT_APPLICATION']
# Доля первоначального взноса относительно суммы кредита
prev['downpayment_ratio'] = prev['AMT_DOWN_PAYMENT'] / prev['AMT_CREDIT']
# Отношение стоимости товара к сумме кредита
prev['goods_credit_ratio'] = prev['AMT_GOODS_PRICE'] / prev['AMT_CREDIT']
# Отношение аннуитетного платежа к сумме кредита
prev['annuity_credit_ratio'] = prev['AMT_ANNUITY'] / prev['AMT_CREDIT']

## Агрегация числовых признаков

Агрегируем числовые характеристики предыдущих заявок по каждому клиенту


In [7]:
# Список числовых признаков для агрегации
num_cols = [
    'AMT_CREDIT',
    'AMT_APPLICATION',
    'AMT_ANNUITY',
    'AMT_GOODS_PRICE',
    'CNT_PAYMENT',
    'DAYS_DECISION',
    'SELLERPLACE_AREA',
    'DAYS_FIRST_DRAWING',
    'DAYS_FIRST_DUE',
    'DAYS_LAST_DUE',
    'DAYS_TERMINATION',
    'credit_to_app_ratio',
    'downpayment_ratio',
    'goods_credit_ratio',
    'annuity_credit_ratio',
]

# Группируем по клиенту и вычисляем статистики по каждому числовому признаку
prev_agg = (
    prev
    .groupby('SK_ID_CURR')[num_cols]
    .agg(['mean', 'max', 'min', 'std', 'sum'])
)

# Формируем названия колонок: признак_статистика
prev_agg.columns = ['_'.join(col) for col in prev_agg.columns]

# Добавляем общее количество предыдущих заявок
prev_agg['TOTAL_PREV_COUNT'] = prev.groupby('SK_ID_CURR').size()

# Объединяем агрегированные признаки с основным датасетом
app = app.merge(prev_agg, how='left',
                left_on='SK_ID_CURR', right_index=True)

### Статусы заявок (долевые признаки)

Рассчитываем долю заявок каждого статуса (одобрена, отказано, отменена и т.д.) для каждого клиента


In [8]:
# Смотрим распределение статусов заявок
prev['NAME_CONTRACT_STATUS'].value_counts()

# Строим кросс-таблицу: клиент x статус заявки
status = pd.crosstab(prev['SK_ID_CURR'], prev['NAME_CONTRACT_STATUS'])
status['TOTAL_PREV_COUNT'] = status.sum(axis=1)

# Вычисляем долю каждого статуса по каждому клиенту
for col_name in ['Approved', 'Refused', 'Canceled', 'Unused offer']:
    status[f'{col_name.upper()}_RATIO'] = (
        status[col_name] / status['TOTAL_PREV_COUNT']
    )

status = status.drop(columns=['TOTAL_PREV_COUNT'])

app = app.merge(status, how='left',
                left_on='SK_ID_CURR', right_index=True)

### Агрегация категориальных признаков

Для каждого клиента рассчитываем долю заявок по каждой категории (тип контракта, канал продаж и т.д.)


In [9]:
# Список категориальных признаков для агрегации
cat_cols = [
    'CODE_REJECT_REASON',      # Причина отказа
    'NAME_CONTRACT_TYPE',      # Тип контракта
    'NAME_PORTFOLIO',          # Портфель
    'NAME_CLIENT_TYPE',        # Тип клиента
    'NAME_YIELD_GROUP',        # Группа доходности
    'CHANNEL_TYPE',            # Канал продаж
    'NAME_PAYMENT_TYPE',       # Тип платежа
]

for col in cat_cols:
    # Строим кросс-таблицу: клиент x категория
    crosstab = pd.crosstab(prev['SK_ID_CURR'], prev[col])
    total = crosstab.sum(axis=1).replace(0, np.nan)
    # Вычисляем долю каждой категории для каждого клиента
    ratios = crosstab.div(total, axis=0)
    ratios = ratios.add_prefix(f'{col}_').add_suffix('_RATIO')
    app = app.merge(ratios, how='left', left_on='SK_ID_CURR', right_index=True)

### Признаки последней заявки

Извлекаем информацию о последней по времени заявке для каждого клиента


In [10]:
# Выбираем последнюю заявку для каждого клиента (по DAYS_DECISION)
last_app = prev.loc[prev.groupby('SK_ID_CURR')['DAYS_DECISION'].idxmax()]

# Отбираем ключевые поля последней заявки
last_features = last_app[[
    'SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'CODE_REJECT_REASON',
    'NAME_CLIENT_TYPE', 'AMT_CREDIT', 'AMT_APPLICATION', 'DAYS_DECISION'
]].copy()

last_features = last_features.rename(columns={
    'NAME_CONTRACT_STATUS': 'LAST_STATUS',
    'CODE_REJECT_REASON': 'LAST_REJECT_REASON',
    'NAME_CLIENT_TYPE': 'LAST_CLIENT_TYPE',
    'AMT_CREDIT': 'LAST_AMT_CREDIT',
    'AMT_APPLICATION': 'LAST_AMT_APPLICATION',
    'DAYS_DECISION': 'LAST_DAYS_DECISION',
})

app = app.merge(last_features, how='left', on='SK_ID_CURR')

### Временные признаки

Создаём признаки, связанные со временем подачи заявок и интервалами между ними


In [11]:
# Размах дат принятия решений по заявкам клиента
app['DAYS_DECISION_range'] = app['DAYS_DECISION_max'] - app['DAYS_DECISION_min']

# Средний интервал между решениями (размах / количество заявок)
app['DAYS_DECISION_mean_gap'] = (
    app['DAYS_DECISION_range'] / app['TOTAL_PREV_COUNT'].replace(0, np.nan)
)

# Количество заявок за последние 6, 12 и 24 месяцев
now = 0
for months, label in [(6, '6M'), (12, '12M'), (24, '24M')]:
    days = -months * 30
    count = (
        prev[prev['DAYS_DECISION'] > days]
        .groupby('SK_ID_CURR').size()
    )
    app[f'prev_count_last_{label}'] = count

### Флаги и пропуски как сигнал

Извлекаем информацию из флагов и пропущенных значений, которые могут быть полезным сигналом


In [12]:
# FLAG_LAST_APPL_PER_CONTRACT — доля заявок, являющихся последними по контракту
flag_crosstab = pd.crosstab(
    prev['SK_ID_CURR'], prev['FLAG_LAST_APPL_PER_CONTRACT']
)
total_flag = flag_crosstab.sum(axis=1).replace(0, np.nan)
flag_ratios = flag_crosstab.div(total_flag, axis=0)
flag_ratios = flag_ratios.add_prefix('FLAG_LAST_APPL_PER_CONTRACT_')
app = app.merge(flag_ratios, how='left',
                left_on='SK_ID_CURR', right_index=True)

# NFLAG_INSURED_ON_APPROVAL — доля застрахованных заявок среди всех заявок клиента
insured = prev[['SK_ID_CURR', 'NFLAG_INSURED_ON_APPROVAL']].copy()
insured['NFLAG_INSURED_ON_APPROVAL'] = (
    insured['NFLAG_INSURED_ON_APPROVAL'].fillna(0)
)
insured_agg = (
    insured.groupby('SK_ID_CURR')['NFLAG_INSURED_ON_APPROVAL'].mean()
)
app['NFLAG_INSURED_ON_APPROVAL_RATIO'] = insured_agg

# NFLAG_LAST_APPL_IN_DAY — доля заявок, бывших последними в день подачи
last_in_day = (
    prev.groupby('SK_ID_CURR')['NFLAG_LAST_APPL_IN_DAY'].mean()
)
app['NFLAG_LAST_APPL_IN_DAY_RATIO'] = last_in_day

# HAS_DOWN_PAYMENT — был ли хотя бы один первоначальный взнос у клиента
downpayment_exists = (
    prev.groupby('SK_ID_CURR')['AMT_DOWN_PAYMENT']
    .apply(lambda x: x.notna().any())
)
app['HAS_DOWN_PAYMENT'] = downpayment_exists.astype(int)

# HAS_DAYS_* — наличие полей DAYS_FIRST_DRAWING и др.
# (отсутствуют для неодобренных заявок, поэтому их наличие — сигнал)
drawing_cols = [
    'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE',
    'DAYS_LAST_DUE', 'DAYS_TERMINATION'
]
for col in drawing_cols:
    missing_flag = (
        prev.groupby('SK_ID_CURR')[col]
        .apply(lambda x: x.notna().any())
    )
    app[f'HAS_{col}'] = missing_flag.astype(int)

## Обработка выбросов


In [ ]:
# Ограничиваем доход на уровне 99-го перцентиля (убираем экстремальные значения) для признака AMT_INCOME_TOTAL
upper = app['AMT_INCOME_TOTAL'].quantile(0.99)
app['AMT_INCOME_TOTAL'] = app['AMT_INCOME_TOTAL'].clip(upper=upper)

# Удаляем аномальные записи с максимальным DAYS_EMPLOYED
app = app[app['DAYS_EMPLOYED'] != app['DAYS_EMPLOYED'].max()]

## Признаки по одобренным заявкам

Выделяем отдельную агрегацию только по одобренным заявкам — это даёт более чистую картину кредитной истории клиента


In [14]:
# Фильтруем только одобренные заявки
approved = prev[
    prev['NAME_CONTRACT_STATUS'] == 'Approved'
].copy()

approved['credit_to_app_ratio'] = (
    approved['AMT_CREDIT'] / approved['AMT_APPLICATION']
)
approved['downpayment_ratio'] = (
    approved['AMT_DOWN_PAYMENT'] / approved['AMT_CREDIT']
)
approved['goods_credit_ratio'] = (
    approved['AMT_GOODS_PRICE'] / approved['AMT_CREDIT']
)
approved['annuity_credit_ratio'] = (
    approved['AMT_ANNUITY'] / approved['AMT_CREDIT']
)

approved_agg = (
    approved
    .groupby('SK_ID_CURR')[num_cols]
    .agg(['mean', 'max', 'min', 'std', 'sum'])
)

approved_agg.columns = [
    'APPROVED_' + '_'.join(col) for col in approved_agg.columns
]

app = app.merge(approved_agg, how='left',
                left_on='SK_ID_CURR', right_index=True)

## Обработка пропусков


In [ ]:
# Заполняем возраст автомобиля нулём (нет машины — нет возраста)
app['OWN_CAR_AGE'] = app['OWN_CAR_AGE'].fillna(0)

## Сохранение результата


In [ ]:
# Сохраняем датасет с новыми признаками
app.to_csv('../data/app_feature_engineered.csv', index=False)

# Заключение

Здесь обработаны фичи и пропуски при помощи некоторых эвристик.

Оставшиеся пропуски будут проходить через препроцессор, в котором будет осуществляться
и заполнение NaN значений и кодировка категориальных признаков.
Это делается во избежание утечки данных про весь датасет при фиче-инжиниринге.
